# 실습 1. K-최근접 이웃으로 악성코드 분류

**과목:** 인공지능과 정보보호 | **제출 형식:** Colab `.ipynb`  
**목표:** 실제 악성코드 데이터셋을 사용해 정상 파일(0)과 악성 파일(1)을 분류하고, **K=1·3·5·10·50**의 테스트 성능을 비교한다.

- 강의자료: `7week_01_악성코드분류와_K최근접이웃` (데이터 전처리·KNN 실습)
- 원본 데이터: https://www.kaggle.com/datasets/nsaravana/malware-detection
- **실행 방법:** Google Colab에서 `런타임 → 모두 실행`을 선택한다. 자동 다운로드가 제한되면 Kaggle에서 받은 CSV를 직접 업로드한다.
- 실제 수치 결과와 그래프는 원본 데이터를 불러온 후 **런타임 → 모두 실행**하여 생성한다.

> 실행 환경: Google Colab (Python 3). 외부 CSV를 원본 데이터셋에서 다운로드하고 학습·평가를 실행한다.

In [ ]:
# 공개 Kaggle 데이터셋 다운로드를 위한 도구를 코랩에 설치한다.
%pip -q install kagglehub

In [ ]:
# 경로를 안전하게 찾기 위해 pathlib을 사용한다.
from pathlib import Path
# 데이터셋을 표로 읽기 위해 pandas를 사용한다.
import pandas as pd
# 공개 Kaggle 데이터셋을 다운로드하는 도구를 불러온다.
import kagglehub

# 강의자료에 명시된 Kaggle 악성코드 데이터셋을 내려받는다.
DATASET_ID = 'nsaravana/malware-detection'
EXPECTED_CSV = 'Malware dataset.csv'
try:
    data_folder = Path(kagglehub.dataset_download(DATASET_ID))
    files_found = list(data_folder.rglob('*.csv'))
    matches = [p for p in files_found if p.name.lower() == EXPECTED_CSV.lower()]
    csv_path = matches[0] if matches else (files_found[0] if len(files_found)==1 else None)
    if csv_path is None:
        raise FileNotFoundError(f'CSV를 특정할 수 없습니다. 발견된 파일: {files_found}')
except Exception as err:
    # 자동 다운로드가 막히면 강의자료의 CSV 파일을 수동 업로드할 수 있다.
    print('자동 다운로드 실패:', type(err).__name__, str(err)[:220])
    print('Kaggle에서 Malware dataset.csv를 받아 업로드하세요.')
    from google.colab import files
    uploaded = files.upload()
    csv_path = Path(next((n for n in uploaded if n.lower().endswith('.csv')), ''))
    if not csv_path.is_file():
        raise FileNotFoundError('CSV 파일 업로드가 필요합니다.')

# 선택한 CSV 파일을 판다스 데이터프레임으로 읽는다.
malware = pd.read_csv(csv_path)
print('데이터 파일:', csv_path.name)
print('원본 데이터 크기(행, 열):', malware.shape)
print(malware.head(3).to_string(index=False))

## 1. 데이터 분석·전처리

강의자료의 `classification`이 정답이며 `hash`는 파일 식별자이므로 모델 입력에서 제외한다. 숫자 피처는 훈련 데이터에서만 평균/표준편차를 학습해 표준화한다. KNN은 거리 기반이므로 특성의 단위 차이가 결과에 영향을 줄 수 있다.

In [ ]:
# 분류 레이블을 올바른 0/1 숫자로 변환한다.
raw_target = malware['classification'].astype(str).str.strip().str.lower()
target = raw_target.map({'benign': 0, 'malware': 1, '0': 0, '1': 1})
if target.isna().any():
    raise ValueError(f'예상하지 못한 classification 값: {raw_target[target.isna()].unique()}')
target = target.astype(int)

# 파일 해시와 정답 열은 학습 입력에 넣지 않는다.
data = malware.drop(columns=['hash','classification'], errors='ignore')
# 숫자형으로 변환할 수 없는 값은 결측으로 두고 훈련 데이터 기준으로 보완한다.
data = data.apply(pd.to_numeric, errors='coerce')
# 전부 결측인 열을 제거한다.
all_empty = data.columns[data.isna().all()].tolist()
if all_empty:
    print('학습 불가 열 제외:', all_empty)
    data = data.drop(columns=all_empty)

print('입력 특성 개수:', data.shape[1])
print('클래스 수 (0=정상, 1=악성):', target.value_counts().to_dict())
print('전체 결측값:', int(data.isna().sum().sum()))
assert data.shape[1] > 0, '수치형 특성이 없습니다.'

# 전체 10만 건으로도 학습할 수 있으나 KNN은 많은 거리 계산이 필요하다.
# 코랩 일반 런타임에서 비교하기 쉽게 기본 20,000개를 층화 추출한다.
# 전체 데이터로 실행하려면 MAX_SAMPLES를 None으로 바꾼다.
MAX_SAMPLES = 20000
if MAX_SAMPLES is not None and len(data) > MAX_SAMPLES:
    from sklearn.model_selection import train_test_split
    data, _, target, _ = train_test_split(
        data, target, train_size=MAX_SAMPLES, random_state=42, stratify=target
    )
    print('계산 시간 절약을 위해 원본에서 층화 추출:', len(data), '개')
else:
    print('전체 레코드 사용:', len(data), '개')

# 훈련 80%와 테스트 20%를 고정 시드로 분리한다.
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42, stratify=target
)
print('훈련 형태:', train_input.shape, '/ 테스트 형태:', test_input.shape)

## 2. KNN 학습: K=1, 3, 5, 10, 50

**KNN**은 새로운 파일과 가까운 K개 데이터를 찾아 다수결로 정상/악성을 결정한다. `SimpleImputer`는 결측값을 훈련 데이터의 중앙값으로 채우고, `StandardScaler`는 훈련 데이터의 평균과 표준편차로 표준화한다. 두 전처리는 `Pipeline` 안에 있어 테스트 데이터를 학습에 사용하지 않는다.

In [ ]:
# KNN·파이프라인과 평가 지표를 불러온다.
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np
import matplotlib.pyplot as plt

# K값마다 정확도와 오탐/미탐을 저장할 리스트를 만든다.
knn_results = []
trained_models = {}
for k in [1, 3, 5, 10, 50]:
    # 결측값 보정 → 표준화 → KNN 순서로 학습한다.
    model = make_pipeline(
        SimpleImputer(strategy='median'),
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    )
    model.fit(train_input, train_target)
    pred = model.predict(test_input)
    # 0=정상, 1=악성 순서로 혼동행렬을 계산한다.
    tn, fp, fn, tp = confusion_matrix(test_target, pred, labels=[0,1]).ravel()
    knn_results.append({
        'K': k,
        '정확도': accuracy_score(test_target,pred),
        '정밀도': precision_score(test_target,pred,zero_division=0),
        '재현율': recall_score(test_target,pred,zero_division=0),
        'F1': f1_score(test_target,pred,zero_division=0),
        '오탐(FP)': int(fp),
        '미탐(FN)': int(fn)
    })
    trained_models[k] = model

# K별 성능을 보기 쉬운 표로 출력한다.
result_df = pd.DataFrame(knn_results)
display(result_df.round(4))

## 3. 평가 결과 그림과 혼동행렬

악성(1)을 정상(0)이라고 예측한 **미탐(FN)**은 실제 피해로 이어질 수 있다. 정상(0)을 악성(1)이라고 예측한 **오탐(FP)**은 불필요한 차단을 만든다.

In [ ]:
# K에 따라 테스트 정확도가 어떻게 달라지는지 그린다.
plt.figure(figsize=(7,4))
plt.plot(result_df['K'], result_df['정확도'], marker='o', label='Test accuracy')
plt.xlabel('K (n_neighbors)')
plt.ylabel('Accuracy')
plt.title('Malware KNN: K vs test accuracy')
plt.grid(alpha=0.3)
plt.show()

# 강의에서 사용한 K=5의 상세 분류 결과를 확인한다.
k5_prediction = trained_models[5].predict(test_input)
print('K=5 혼동행렬 (행=실제 0/1, 열=예측 0/1):')
print(confusion_matrix(test_target, k5_prediction, labels=[0,1]))
print(classification_report(test_target, k5_prediction,
      labels=[0,1], target_names=['benign(0)', 'malware(1)'], zero_division=0))

## 4. 두 줄 결과 해석 (실제 실행값에 따라 자동 생성)

강의자료의 과제는 "K에 따라 정확도가 어떻게 달라지는지 두 줄로 쓰기"다. 코드를 실행해 만든 아래 문장을 자신의 관찰과 일치하는지 확인한 뒤 수정하면 된다.

In [ ]:
# 실제 테스트 정확도에서 최댓값과 최솟값을 찾아 수치로 설명한다.
high = result_df.loc[result_df['정확도'].idxmax()]
low = result_df.loc[result_df['정확도'].idxmin()]
print(f"1. 이번 실험에서는 K={int(high['K'])}일 때 정확도 {high['정확도']:.2%}로 가장 높고, "
      f"K={int(low['K'])}일 때 {low['정확도']:.2%}로 가장 낮았다.")
print('2. K가 너무 작으면 가까운 한두 개의 예외 데이터에 민감하고, '
      '너무 크면 다른 클래스까지 많이 포함해 분류 경계가 지나치게 단순해질 수 있다.')
print('※ 테스트 세트에서 K를 반복 선택하면 평가 편향이 생길 수 있다. 실제 튜닝은 검증/교차 검증으로 수행한다.')

## 5. 직접 확인해야 할 사항

- 파일이 정말 `Malware dataset.csv`인지, 레이블이 정상 0·악성 1로 변환되었는지 확인한다.
- 20,000개 표본 실험 결과와 전체 100,000개 실험 결과는 서로 다를 수 있다.
- KNN은 **악성코드 데이터**로 실험했으며, 다른 피싱사이트 데이터로 만든 모델과 정확도를 직접 비교해 우열을 단정하지 않는다.
- 코랩에서 `파일 → 다운로드 → .ipynb 다운로드`를 선택해 제출한다.